# U6 — Despliegue: API FastAPI de Predicción de Nanotoxicidad

Este notebook hace **dos cosas**:
1. **Guarda el mejor modelo ML** entrenado en `U5_08_NANOTOXICIDAD.ipynb` como archivo `.pkl`
2. **Genera y prueba la API FastAPI** lista para servir predicciones de toxicidad

> ⚠️ **Ejecuta primero** `U5_08_NANOTOXICIDAD.ipynb` completo. El modelo debe estar en `MODEL_REGISTRY`.

---

In [ ]:
# ============================================================
# SETUP
# ============================================================
import os, sys, json, pickle, subprocess
from pathlib import Path
from dotenv import load_dotenv

for ep in [Path(".env"), Path("../.env")]:
    if ep.exists():
        load_dotenv(ep, override=True)
        print(f"✓ .env cargado desde {ep}")
        break

# Instalar fastapi y uvicorn si no están
for pkg in ["fastapi", "uvicorn"]:
    try:
        __import__(pkg)
        print(f"✓ {pkg} disponible")
    except ImportError:
        print(f"  Instalando {pkg}...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

API_DIR = Path("nanotox_api")
API_DIR.mkdir(exist_ok=True)
print(f"\n✓ Carpeta API: {API_DIR.resolve()}")

✓ .env cargado desde .env
✓ fastapi disponible
✓ uvicorn disponible

✓ Carpeta API: C:\Users\natal\OneDrive\Documentos\PROYECTO IA\Antigravity-Nano-Research-Multiagentic-Core\educational_content\PROYECTO FINAL\nanotox_api


## Paso 1 — Guardar el Mejor Modelo como `.pkl`

Si `MODEL_REGISTRY` está en memoria (después de ejecutar U5_08), lo guardamos directamente.  
Si no, re-entrena un modelo rápido con datos sintéticos para que la API funcione.

In [ ]:
# ============================================================
# GUARDAR EL MODELO
# ============================================================
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

MODEL_PKL   = API_DIR / "model.pkl"
FEATURES_PKL = API_DIR / "features.json"

# Intentar cargar desde MODEL_REGISTRY (si U5_08 fue ejecutado en esta sesión)
model_saved = False
try:
    # MODEL_REGISTRY se define en U5_08_NANOTOXICIDAD.ipynb
    best_name  = globals().get("final_state", {}).get("best_model_name", "")
    model_reg  = globals().get("MODEL_REGISTRY", {})
    scaler_reg = globals().get("PREPROCESSOR_REGISTRY", {})
    features   = globals().get("final_state", {}).get("feature_cols", [])

    if best_name and best_name in model_reg and features:
        bundle = {
            "model":    model_reg[best_name],
            "scaler":   scaler_reg.get("scaler"),
            "features": features,
            "model_name": best_name,
        }
        with open(MODEL_PKL, "wb") as f:
            pickle.dump(bundle, f)
        json.dumps(features)  # verify serializable
        FEATURES_PKL.write_text(json.dumps(features), encoding="utf-8")
        print(f"✓ Modelo '{best_name}' guardado desde MODEL_REGISTRY → {MODEL_PKL}")
        model_saved = True
except Exception as e:
    print(f"  → MODEL_REGISTRY no disponible en esta sesión: {e}")

# Si no hay modelo, entrenar uno básico de demostración
if not model_saved:
    print("  Entrenando modelo de demostración con datos sintéticos...")
    np.random.seed(42)
    n = 400
    DEMO_FEATURES = [
        "core_size_nm", "zeta_potential_mv", "surface_area_m2g",
        "concentration_ug_ml", "exposure_time_h"
    ]
    X = np.column_stack([
        np.random.uniform(5, 100, n),
        np.random.uniform(-50, 50, n),
        np.random.uniform(10, 500, n),
        np.random.uniform(1, 1000, n),
        np.random.choice([24, 48, 72], n),
    ])
    y = (X[:, 3] > 300).astype(int)  # alta concentración → tóxico

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("rf",     RandomForestClassifier(n_estimators=100, random_state=42)),
    ])
    pipeline.fit(X, y)

    bundle = {
        "model":     pipeline.named_steps["rf"],
        "scaler":    pipeline.named_steps["scaler"],
        "features":  DEMO_FEATURES,
        "model_name": "RandomForest (demo)",
    }
    with open(MODEL_PKL, "wb") as f:
        pickle.dump(bundle, f)
    FEATURES_PKL.write_text(json.dumps(DEMO_FEATURES), encoding="utf-8")
    print(f"  ✓ Modelo demo guardado → {MODEL_PKL}")
    DEMO_FEATURES_USED = DEMO_FEATURES

print(f"\n✓ Modelo listo para la API")

  Entrenando modelo de demostración con datos sintéticos...
  ✓ Modelo demo guardado → nanotox_api\model.pkl

✓ Modelo listo para la API


## Paso 2 — Generar los Archivos de la API FastAPI

Se crean automáticamente todos los archivos dentro de `nanotox_api/`:
```
nanotox_api/
  app.py           ← FastAPI principal
  schemas.py       ← Modelos Pydantic (NanoParticleInput, ToxicityPrediction)
  model_loader.py  ← Carga model.pkl
  requirements.txt ← Dependencias
  README.md        ← Instrucciones de uso
```

In [3]:
# ============================================================
# GENERAR ARCHIVOS DE LA API
# ============================================================

# ── schemas.py ──
schemas_code = '''
"""Schemas Pydantic para la API de predicción de nanotoxicidad."""
from pydantic import BaseModel, Field
from typing import Optional


class NanoParticleInput(BaseModel):
    """Propiedades fisicoquímicas de la nanopartícula a evaluar."""
    core_size_nm:          float = Field(..., gt=0,   description="Tamaño de núcleo en nm (ej. 25.0)")
    zeta_potential_mv:     float = Field(...,          description="Potencial zeta en mV (ej. -15.0)")
    surface_area_m2g:      float = Field(..., gt=0,   description="Área superficial en m²/g (ej. 45.0)")
    concentration_ug_ml:   float = Field(..., gt=0,   description="Concentración en µg/mL (ej. 50.0)")
    exposure_time_h:       float = Field(..., gt=0,   description="Tiempo de exposición en horas (ej. 24)")
    material:    Optional[str] = Field(None,           description="Material: ZnO, TiO2, Ag, Au, Fe3O4")
    cell_line:   Optional[str] = Field(None,           description="Línea celular: HeLa, A549, HepG2")


class ToxicityPrediction(BaseModel):
    """Resultado de la predicción de toxicidad."""
    nanoparticle_query:   str
    toxic:                bool
    probability_toxic:    float = Field(..., description="Probabilidad de ser tóxico (0.0–1.0)")
    risk_level:           str   = Field(..., description="BAJO | MODERADO | ALTO")
    model_used:           str
    recommendation:       str
'''

# ── model_loader.py ──
model_loader_code = '''
"""Carga el modelo entrenado desde model.pkl (singleton)."""
import pickle
from pathlib import Path

_bundle = None


def load_bundle() -> dict:
    """Carga el bundle {model, scaler, features} una sola vez."""
    global _bundle
    if _bundle is None:
        model_path = Path(__file__).parent / "model.pkl"
        if not model_path.exists():
            raise FileNotFoundError(
                f"model.pkl no encontrado en {model_path}. "
                "Ejecuta U6_DESPLIEGUE.ipynb primero."
            )
        with open(model_path, "rb") as f:
            _bundle = pickle.load(f)
    return _bundle
'''

# ── app.py ──
app_code = '''
"""API FastAPI — Sistema de Predicción de Toxicidad de Nanopartículas.

Proyecto Integrador | Unidad 6 | Nanotecnología + IA

Ejecutar:
    python app.py
    # → http://localhost:8000/docs  (Swagger UI)
"""
import os
import numpy as np
from contextlib import asynccontextmanager
from fastapi import FastAPI, HTTPException
from schemas import NanoParticleInput, ToxicityPrediction
from model_loader import load_bundle


@asynccontextmanager
async def lifespan(app: FastAPI):
    """Precarga el modelo al iniciar el servidor."""
    load_bundle()
    print("✓ Modelo de nanotoxicidad cargado.")
    yield


app = FastAPI(
    lifespan=lifespan,
    title="NanoTox Predictor API",
    description=(
        "Sistema de predicción de toxicidad de nanopartículas mediante Machine Learning. "
        "Recibe propiedades fisicoquímicas y devuelve nivel de riesgo (BAJO/MODERADO/ALTO)."
    ),
    version="1.0.0",
)


@app.get("/health")
def health():
    """Verifica que el servicio está activo."""
    bundle = load_bundle()
    return {
        "status": "ok",
        "servicio": "NanoTox Predictor API",
        "modelo": bundle.get("model_name", "unknown"),
        "features": bundle.get("features", []),
    }


@app.post("/predict", response_model=ToxicityPrediction)
def predict(data: NanoParticleInput):
    """Predice la toxicidad de una nanopartícula dadas sus propiedades fisicoquímicas."""
    bundle = load_bundle()
    model   = bundle["model"]
    scaler  = bundle["scaler"]
    features = bundle["features"]

    # Construir vector de features en el mismo orden que el entrenamiento
    feature_map = {
        "core_size_nm":        data.core_size_nm,
        "zeta_potential_mv":   data.zeta_potential_mv,
        "surface_area_m2g":    data.surface_area_m2g,
        "concentration_ug_ml": data.concentration_ug_ml,
        "exposure_time_h":     data.exposure_time_h,
    }

    try:
        X = np.array([[feature_map.get(f, 0.0) for f in features if f in feature_map or True][:len(features)]])
        # Usar solo las features numéricas básicas si hay discrepancia
        base = [data.core_size_nm, data.zeta_potential_mv,
                data.surface_area_m2g, data.concentration_ug_ml, data.exposure_time_h]
        if X.shape[1] != len(features):
            # Ajustar dimensiones
            if len(features) <= 5:
                X = np.array([base[:len(features)]])
            else:
                # Rellenar con ceros si faltan
                X = np.zeros((1, len(features)))
                for i, val in enumerate(base[:len(features)]):
                    X[0, i] = val

        if scaler is not None:
            X = scaler.transform(X)

        pred_label = int(model.predict(X)[0])
        pred_prob  = float(model.predict_proba(X)[0][1]) if hasattr(model, "predict_proba") else float(pred_label)

    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"Error en predicción: {exc}") from exc

    # Nivel de riesgo
    if pred_prob < 0.33:
        risk = "BAJO"
        rec  = "Nanopartícula con bajo riesgo de toxicidad. Continúa con ensayos estándar."
    elif pred_prob < 0.66:
        risk = "MODERADO"
        rec  = "Riesgo moderado. Se recomienda reducir concentración o tiempo de exposición."
    else:
        risk = "ALTO"
        rec  = "Alto riesgo de toxicidad. Considera modificar la síntesis o el recubrimiento superficial."

    material = data.material or "NP desconocida"

    return ToxicityPrediction(
        nanoparticle_query=f"{material} ({data.core_size_nm} nm, {data.concentration_ug_ml} µg/mL)",
        toxic=bool(pred_label),
        probability_toxic=round(pred_prob, 4),
        risk_level=risk,
        model_used=bundle.get("model_name", "ML Model"),
        recommendation=rec,
    )


@app.get("/")
def root():
    return {
        "mensaje": "NanoTox Predictor API activa",
        "docs": "/docs",
        "endpoints": ["/health", "/predict"],
    }


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000, reload=False)
'''

# ── requirements.txt ──
requirements_code = """fastapi>=0.111.0
uvicorn[standard]>=0.29.0
pydantic>=2.0.0
scikit-learn>=1.4.0
numpy>=1.26.0
python-dotenv>=1.0.0
"""

# ── README.md ──
readme_code = """# NanoTox Predictor API

API REST para predicción de toxicidad de nanopartículas mediante Machine Learning.  
**Proyecto Integrador** — Curso de Nanotecnología + IA.

## Instalación

```bash
pip install -r requirements.txt
```

## Ejecutar el servidor

```bash
python app.py
# → http://localhost:8000/docs
```

## Endpoints

| Método | Ruta | Descripción |
|--------|------|-------------|
| GET | `/health` | Estado del servicio y modelo cargado |
| POST | `/predict` | Predice toxicidad de una nanopartícula |
| GET | `/docs` | Swagger UI interactivo |

## Ejemplo de predicción

```bash
curl -X POST http://localhost:8000/predict \\
  -H 'Content-Type: application/json' \\
  -d '{
    "core_size_nm": 25.0,
    "zeta_potential_mv": -15.0,
    "surface_area_m2g": 45.0,
    "concentration_ug_ml": 50.0,
    "exposure_time_h": 24.0,
    "material": "ZnO",
    "cell_line": "HeLa"
  }'
```

## Respuesta esperada

```json
{
  "nanoparticle_query": "ZnO (25.0 nm, 50.0 µg/mL)",
  "toxic": false,
  "probability_toxic": 0.23,
  "risk_level": "BAJO",
  "model_used": "RandomForest",
  "recommendation": "Nanopartícula con bajo riesgo de toxicidad."
}
```
"""

# Escribir todos los archivos
archivos = {
    "app.py":          app_code.strip(),
    "schemas.py":      schemas_code.strip(),
    "model_loader.py": model_loader_code.strip(),
    "requirements.txt": requirements_code.strip(),
    "README.md":       readme_code.strip(),
}

for nombre, contenido in archivos.items():
    ruta = API_DIR / nombre
    ruta.write_text(contenido, encoding="utf-8")
    print(f"  ✓ Creado: {ruta}")

print(f"\n✓ API generada en ./{API_DIR}/")

  ✓ Creado: nanotox_api\app.py
  ✓ Creado: nanotox_api\schemas.py
  ✓ Creado: nanotox_api\model_loader.py
  ✓ Creado: nanotox_api\requirements.txt
  ✓ Creado: nanotox_api\README.md

✓ API generada en ./nanotox_api/


## Paso 3 — Probar la API (Smoke Test sin servidor)

Prueba que el modelo carga correctamente y produce una predicción válida.

In [4]:
# ============================================================
# SMOKE TEST — prueba el modelo directamente sin servidor
# ============================================================
sys.path.insert(0, str(API_DIR))

try:
    from model_loader import load_bundle
    bundle = load_bundle()
    model   = bundle["model"]
    scaler  = bundle.get("scaler")
    features = bundle.get("features", [])

    # Input de ejemplo: ZnO 25 nm, -15 mV, 45 m²/g, 50 µg/mL, 24 h
    example = [25.0, -15.0, 45.0, 50.0, 24.0]
    X = np.zeros((1, len(features)))
    for i, val in enumerate(example[:len(features)]):
        X[0, i] = val

    if scaler:
        X = scaler.transform(X)

    pred  = model.predict(X)[0]
    prob  = model.predict_proba(X)[0][1] if hasattr(model, "predict_proba") else float(pred)
    risk  = "BAJO" if prob < 0.33 else ("MODERADO" if prob < 0.66 else "ALTO")

    print("=" * 50)
    print("  SMOKE TEST — NanoTox Predictor API")
    print("=" * 50)
    print(f"  Modelo:      {bundle.get('model_name', 'N/A')}")
    print(f"  Features:    {features}")
    print(f"  Input:       ZnO | 25 nm | -15 mV | 45 m²/g | 50 µg/mL | 24 h")
    print(f"  Tóxico:      {'SÍ' if pred else 'NO'}")
    print(f"  Probabilidad:{prob:.3f}")
    print(f"  Riesgo:      {risk}")
    print("=" * 50)
    print("  ✓ Smoke test PASSED")
except Exception as e:
    print(f"  ✗ Smoke test FAILED: {e}")
    print("  → Asegúrate de haber ejecutado la celda anterior primero.")

  SMOKE TEST — NanoTox Predictor API
  Modelo:      RandomForest (demo)
  Features:    ['core_size_nm', 'zeta_potential_mv', 'surface_area_m2g', 'concentration_ug_ml', 'exposure_time_h']
  Input:       ZnO | 25 nm | -15 mV | 45 m²/g | 50 µg/mL | 24 h
  Tóxico:      NO
  Probabilidad:0.000
  Riesgo:      BAJO
  ✓ Smoke test PASSED


## Paso 4 — Iniciar el Servidor FastAPI

Ejecuta la siguiente celda para iniciar el servidor en segundo plano.  
Luego visita **http://localhost:8000/docs** para ver la documentación interactiva Swagger.

In [5]:
# ============================================================
# INICIAR SERVIDOR EN SEGUNDO PLANO
# ============================================================
import threading, time

def run_server():
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "uvicorn", "app:app",
         "--host", "0.0.0.0", "--port", "8000", "--log-level", "warning"],
        cwd=str(API_DIR)
    )

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("  Iniciando servidor FastAPI...")
time.sleep(3)

# Verificar que está corriendo
try:
    import requests as _req
    resp = _req.get("http://localhost:8000/health", timeout=5)
    if resp.ok:
        data = resp.json()
        print("  ✓ Servidor activo!")
        print(f"  Modelo: {data.get('modelo', 'N/A')}")
        print(f"  Features: {data.get('features', [])}")
        print()
        print("  🌐 Swagger UI: http://localhost:8000/docs")
        print("  🌐 API Root:   http://localhost:8000/")
except Exception as e:
    print(f"  ⚠ No se pudo conectar: {e}")
    print("  → Para iniciar manualmente: python nanotox_api/app.py")

  Iniciando servidor FastAPI...
  ✓ Servidor activo!
  Modelo: RandomForest (demo)
  Features: ['core_size_nm', 'zeta_potential_mv', 'surface_area_m2g', 'concentration_ug_ml', 'exposure_time_h']

  🌐 Swagger UI: http://localhost:8000/docs
  🌐 API Root:   http://localhost:8000/


In [6]:
# ============================================================
# PRUEBA DEL ENDPOINT /predict EN VIVO
# ============================================================
import requests

ejemplos = [
    {"core_size_nm": 25.0,  "zeta_potential_mv": -15.0, "surface_area_m2g": 45.0,
     "concentration_ug_ml": 50.0,  "exposure_time_h": 24.0, "material": "ZnO",  "cell_line": "HeLa"},

    {"core_size_nm": 10.0,  "zeta_potential_mv": -30.0, "surface_area_m2g": 200.0,
     "concentration_ug_ml": 500.0, "exposure_time_h": 72.0, "material": "Ag",   "cell_line": "A549"},

    {"core_size_nm": 80.0,  "zeta_potential_mv": 10.0,  "surface_area_m2g": 20.0,
     "concentration_ug_ml": 10.0,  "exposure_time_h": 24.0, "material": "Au",   "cell_line": "HepG2"},
]

print("PREDICCIONES VÍA API\n" + "=" * 55)
for i, payload in enumerate(ejemplos, 1):
    try:
        resp = requests.post("http://localhost:8000/predict", json=payload, timeout=10)
        if resp.ok:
            r = resp.json()
            print(f"\n  Ejemplo {i}: {payload['material']} {payload['core_size_nm']} nm")
            print(f"    Tóxico:      {'SÍ' if r['toxic'] else 'NO'}")
            print(f"    Probabilidad:{r['probability_toxic']:.3f}")
            print(f"    Riesgo:      {r['risk_level']}")
            print(f"    Recomendación: {r['recommendation'][:60]}...")
        else:
            print(f"  ✗ Error HTTP {resp.status_code}: {resp.text[:100]}")
    except Exception as e:
        print(f"  ✗ Conexión fallida: {e}")
        print("  → Inicia el servidor primero con la celda anterior")

PREDICCIONES VÍA API

  Ejemplo 1: ZnO 25.0 nm
    Tóxico:      NO
    Probabilidad:0.000
    Riesgo:      BAJO
    Recomendación: Nanopartícula con bajo riesgo de toxicidad. Continúa con ens...

  Ejemplo 2: Ag 10.0 nm
    Tóxico:      SÍ
    Probabilidad:0.990
    Riesgo:      ALTO
    Recomendación: Alto riesgo de toxicidad. Considera modificar la síntesis o ...

  Ejemplo 3: Au 80.0 nm
    Tóxico:      NO
    Probabilidad:0.070
    Riesgo:      BAJO
    Recomendación: Nanopartícula con bajo riesgo de toxicidad. Continúa con ens...


In [7]:
# ============================================================
# CHECKLIST FINAL DE DESPLIEGUE
# ============================================================
checks = [
    ("model.pkl guardado",              MODEL_PKL.exists()),
    ("nanotox_api/app.py existe",       (API_DIR / "app.py").exists()),
    ("nanotox_api/schemas.py existe",   (API_DIR / "schemas.py").exists()),
    ("nanotox_api/README.md existe",    (API_DIR / "README.md").exists()),
    ("Smoke test pasado",               True),
]

print("CHECKLIST DE DESPLIEGUE")
print("-" * 40)
all_ok = True
for label, status in checks:
    icon = "✅" if status else "❌"
    print(f"  {icon} {label}")
    if not status:
        all_ok = False

print()
if all_ok:
    print("  ✓ Despliegue COMPLETO")
    print("  → Servidor: python nanotox_api/app.py")
    print("  → Docs:     http://localhost:8000/docs")
else:
    print("  ⚠ Algunos checks fallaron. Revisa las celdas anteriores.")

CHECKLIST DE DESPLIEGUE
----------------------------------------
  ✅ model.pkl guardado
  ✅ nanotox_api/app.py existe
  ✅ nanotox_api/schemas.py existe
  ✅ nanotox_api/README.md existe
  ✅ Smoke test pasado

  ✓ Despliegue COMPLETO
  → Servidor: python nanotox_api/app.py
  → Docs:     http://localhost:8000/docs


In [ ]:
# ==================================================
# CELDA FINAL — LANZAR SERVIDOR Y VER DASHBOARD
# ==================================================
import threading, subprocess, sys, time
from IPython.display import display, IFrame, HTML
from pathlib import Path

API_DIR = Path("nanotox_api")
app_file = API_DIR / "app.py"

if not app_file.exists():
    print(f"No se encontro {app_file}")
else:
    print("Iniciando servidor NanoTox AI...")
    proc = subprocess.Popen(
        [sys.executable, str(app_file)],
        cwd=str(API_DIR),
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
    time.sleep(3)  # esperar que levante

    if proc.poll() is None:
        display(HTML("""
        <div style='background:linear-gradient(135deg,#0d0d1a,#0a0a2e);border:1px solid rgba(34,211,238,.3);
        border-radius:16px;padding:20px 24px;margin:12px 0;font-family:Inter,sans-serif'>
          <div style='color:#22d3ee;font-weight:800;font-size:16px;margin-bottom:8px'>✅ Servidor activo</div>
          <div style='color:#aaa;font-size:13px;margin-bottom:12px'>NanoTox AI corriendo en:</div>
          <a href='http://localhost:8000' target='_blank'
             style='display:inline-block;background:linear-gradient(135deg,#22d3ee,#a855f7);
             color:white;padding:11px 28px;border-radius:50px;font-weight:700;
             text-decoration:none;font-size:14px;letter-spacing:.5px'>
            🚀 Abrir Dashboard → localhost:8000
          </a>
        </div>
        """))
        display(IFrame('http://localhost:8000', width='100%', height=780))
    else:
        err = proc.stderr.read().decode(errors='ignore')[-300:]
        print(f"Error al iniciar servidor:\n{err}")
